In [ ]:
!pip install streamlit joblib tensorflow scikit-learn pandas numpy
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴
changed 22 packages in 2s
⠴
⠴3 packages are looking for funding
⠴  run `npm fund` for details
⠴

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import datetime

# --- Page Configuration & Styling ---
st.set_page_config(
    page_title="AeroShield AI | Environmental Cybernetics Auditing",
    layout="wide",
    initial_sidebar_state="expanded"
)

# Custom CSS for UI Enhancement
st.markdown("""
    <style>
    .reportview-container { background: #f5f7f9; }
    .main .block-container { padding-top: 2rem; }
    div.stButton > button:first-child { border-radius: 8px; font-weight: 600; }
    .metric-card {
        background-color: white; padding: 20px; border-radius: 12px;
        box-shadow: 0 4px 6px rgba(0,0,0,0.05); border-left: 5px solid #1E88E5;
    }
    </style>
""", unsafe_allow_html=True)

# --- Authentication Layer (Enhanced for Login / Sign Up Toggle) ---
if "authenticated" not in st.session_state:
    st.session_state.authenticated = False
if "prediction_history" not in st.session_state:
    st.session_state.prediction_history = []
if "user_db" not in st.session_state:
    st.session_state.user_db = {
        "admin": "audit2026"  # Seed default master auditor profile
    }

def logout_user():
    st.session_state.authenticated = False
    st.session_state.prediction_history = []
    st.rerun()

# --- GATEWAY SCREEN ---
if not st.session_state.authenticated:
    st.title("🔒 AeroShield AI Compliance Portal")
    st.markdown("Authenticate or register a profile vector to unlock the evaluation metrics pipeline.")

    col1, _ = st.columns([1.2, 2])
    with col1:
        # Toggle component between Sign In mode and New User Registration mode
        auth_mode = st.radio("Access Strategy Selection:", ["Log In to Account", "Create New Account"], horizontal=True)

        with st.form("Auth Form"):
            username = st.text_input("Username / Auditor ID", placeholder="e.g., admin").strip()
            password = st.text_input("Security Passphrase", type="password", placeholder="••••••••").strip()

            if auth_mode == "Log In to Account":
                submit_auth = st.form_submit_button("Verify & Authenticate", use_container_width=True, type="primary")
                if submit_auth:
                    if username in st.session_state.user_db and st.session_state.user_db[username] == password:
                        st.session_state.authenticated = True
                        st.session_state.username = username
                        st.toast(f"Access granted for user: {username}", icon="🔓")
                        st.rerun()
                    else:
                        st.error("Authentication rejected. Invalid credential mapping values.")

            elif auth_mode == "Create New Account":
                submit_auth = st.form_submit_button("Register Secure Account", use_container_width=True, type="primary")
                if submit_auth:
                    if not username or not password:
                        st.warning("Fields cannot rest empty. Provide uniform input configurations.")
                    elif username in st.session_state.user_db:
                        st.error("Identity Vector Collision: This username is already registered.")
                    else:
                        # Dynamically append new key-value pair directly into the local session context
                        st.session_state.user_db[username] = password
                        st.success(f"Registration successful for `{username}`! Select 'Log In to Account' above to sign in.")
                        st.toast("Profile successfully provisioned!", icon="✨")

    st.stop() # Halts app pipeline rendering until authentication matrix evaluates to true

# --- MAIN APP ASSETS (ONLY ACCESSED IF LOGGED IN) ---
@st.cache_resource
def load_ml_assets():
    xgb_model = joblib.load('xgboost_aqi_model.pkl')
    scaler_X = joblib.load('scaler_X.pkl')
    scaler_y = joblib.load('scaler_y.pkl')
    return xgb_model, scaler_X, scaler_y

try:
    xgb_model, scaler_X, scaler_y = load_ml_assets()
    assets_loaded = True
except Exception as e:
    st.error(f"⚠️ Infrastructure Initialization Error: {e}")
    st.info("Ensure scaler_X.pkl, scaler_y.pkl, and xgboost_aqi_model.pkl match the application root paths.")
    assets_loaded = False

# --- UI Header & Sidebar Controls ---
st.sidebar.markdown(f"**🟢 Session User:** `{st.session_state.username}`")
if st.sidebar.button("Secure Logout", type="secondary"):
    logout_user()

st.title("🌱 Real-Time Air Quality Forecasting System")
st.markdown("Predict future Carbon Monoxide ($CO$) concentrations dynamically using multi-sensor environmental parameters.")
st.divider()

if assets_loaded:
    # --- Advanced Features: Multi-Scenario Testing Panel ---
    st.sidebar.header("🕹️ Pipeline Controls")
    model_choice = st.sidebar.selectbox(
        "Forecasting Engine Architecture:",
        ["XGBoost Regressor (Tabular Anomaly Route)", "CNN-BiLSTM Network (Sequential Context Route)"]
    )

    # Quick Simulation Presets
    preset = st.sidebar.selectbox(
        "Load Environmental Preset Scenario:",
        ["Manual Adjustment", "Clean Baseline", "Heavy Industrial Smog", "Midday Traffic Spike"]
    )

    # Mapping Presets to Sensor Values (S1, S2, S3, S5, T, RH, AH)
    presets = {
        "Manual Adjustment": [1000, 900, 700, 1000, 22.0, 50.0, 1.1],
        "Clean Baseline": [650, 500, 1200, 500, 15.0, 35.0, 0.6],
        "Heavy Industrial Smog": [1650, 1500, 400, 1700, 38.0, 85.0, 2.5],
        "Midday Traffic Spike": [1300, 1100, 550, 1250, 28.0, 45.0, 1.4]
    }

    current_preset = presets[preset]

    st.sidebar.header("📡 Environmental Sensor Array")
    pt08_s1 = st.sidebar.slider("PT08.S1 (CO Sensor Signal)", 500, 2000, current_preset[0])
    pt08_s2 = st.sidebar.slider("PT08.S2 (NMHC Sensor Signal)", 400, 2000, current_preset[1])
    pt08_s3 = st.sidebar.slider("PT08.S3 (NOx Sensor Signal)", 300, 2000, current_preset[2])
    pt08_s5 = st.sidebar.slider("PT08.S5 (O3 Sensor Signal)", 300, 2000, current_preset[3])

    st.sidebar.header("🌤️ Meteorological Constraints")
    temperature = st.sidebar.slider("Ambient Temperature (°C)", -5.0, 50.0, current_preset[4])
    relative_humidity = st.sidebar.slider("Relative Humidity (%)", 0.0, 100.0, current_preset[5])
    absolute_humidity = st.sidebar.slider("Absolute Humidity (AH)", 0.1, 3.0, current_preset[6])

    # --- Main Dashboard Split View ---
    tabs = st.tabs(["🔮 Real-Time Inference Console", "📊 Audit Log & Analytics"])

    with tabs[0]:
        col1, col2 = st.columns([7, 5])

        with col1:
            st.subheader("📋 Ingested Sensor Telemetry Vector")
            input_df = pd.DataFrame([{
                "PT08.S1(CO)": pt08_s1, "PT08.S2(NMHC)": pt08_s2, "PT08.S3(NOx)": pt08_s3,
                "PT08.S5(O3)": pt08_s5, "T (°C)": temperature, "RH (%)": relative_humidity, "AH": absolute_humidity
            }])

            st.dataframe(
                input_df.style.highlight_max(axis=1, props="background-color: #E3F2FD; color: #0c111d;"),
                use_container_width=True
            )

            # Sub-Index Threat Gauge Breakdown
            st.markdown("### 🔍 Feature Importance & Weight Analysis")
            metrics_cols = st.columns(4)
            metrics_cols[0].metric("Hydrocarbon Index", f"{pt08_s2} Hz", delta="Target Vector" if pt08_s2 > 1000 else None, delta_color="inverse")
            metrics_cols[1].metric("Nitrogen Load", f"{pt08_s3} Hz", delta="Low Inversion" if pt08_s3 < 600 else None)
            metrics_cols[2].metric("Photochemical O3", f"{pt08_s5} Hz", delta="Critical Risk" if pt08_s5 > 1400 else None, delta_color="inverse")
            metrics_cols[3].metric("Thermal Metric", f"{temperature}°C")

        with col2:
            st.subheader("⚙️ Executive Pipeline Decision")
            if st.button("Execute Forecasting Pipeline", type="primary", use_container_width=True):
                with st.spinner("Processing architectural vector slicing..."):

                    try:
                        # 1. Vector Formatting
                        raw_input = np.array([[pt08_s1, pt08_s2, pt08_s3, pt08_s5, temperature, relative_humidity, absolute_humidity]])

                        # 2. Mathematical Normalization Guardrail
                        scaled_input = scaler_X.transform(raw_input)

                        if "XGBoost" in model_choice:
                            scaled_prediction = xgb_model.predict(scaled_input).reshape(-1, 1)
                        else:
                            import tensorflow as tf
                            dl_model = tf.keras.models.load_model('aqi_deep_learning_model.h5', compile=False)

                            # FIXED: Broadcast the 2D scaled row into a 3D tensor sequence (1, 24, 7)
                            # to prevent the recurrent cells from dropping into a flat zero state vector.
                            time_steps = 24
                            sequenced_features = np.repeat(scaled_input[:, np.newaxis, :], time_steps, axis=1)
                            scaled_prediction = dl_model.predict(sequenced_features)

                        # 3. Target Inverse Boundary Projection
                        # FIXED: Ensured output scalar unpacking and proper variable assignment mapping
                        final_prediction = scaler_y.inverse_transform(scaled_prediction)[0][0]
                        co_value = max(0.0, float(final_prediction))

                        # Alert Threshold Metrics Logic (No generic color text strings)
                        if co_value < 1.5:
                            risk_tier = "Low / Nominal Risk Threshold"
                            alert_title = "Clean Ambient Level (Healthy Environment)"
                            alert_desc = "Atmospheric concentration indexes indicate clean atmospheric baselines. No current ventilation throttling or hazard mitigations are required."
                        elif co_value < 3.0:
                            risk_tier = "Elevated Caution Threshold"
                            alert_title = "Moderate Air Level (Sensor Aggregation Alert)"
                            alert_desc = "Elevated baseline values identified. Potential respiratory risk indices detected for highly vulnerable or sensitive population sectors."
                        else:
                            risk_tier = "Critical Hazard Alert"
                            alert_title = "High Atmospheric Pollution Warning"
                            alert_desc = "Critical particulate thresholds crossed. Immediate industrial scrubbing, structural air extraction, or environmental mitigation procedures required."

                        # Render UI Metric Card
                        st.markdown(f"""
                            <div class="metric-card">
                                <h4 style="margin:0; color:#555;">Target Forecast Output:</h4>
                                <h1 style="margin:5px 0; color:#1E88E5;">{co_value:.3f} <span style="font-size:18px; color:#777;">mg/m³</span></h1>
                                <p style="margin:0; font-weight:600; color:#e056fd;">⚠️ Status Assessment: {risk_tier}</p>
                            </div>
                        """, unsafe_allow_html=True)

                        st.warning(f"**{alert_title}**\n\n{alert_desc}")

                        # Log Results dynamically into state cache for analytics tab
                        st.session_state.prediction_history.append({
                            "Timestamp": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                            "Architecture": model_choice,
                            "CO Forecast (mg/m³)": round(co_value, 4),
                            "Risk Assessment": risk_tier,
                            "Temp": temperature,
                            "Humidity": relative_humidity
                        })

                    except Exception as pipeline_err:
                        st.error(f"Execution Error inside Inference Pipeline: {pipeline_err}")

    with tabs[1]:
        st.subheader("📁 Historical Pipeline Verification Audit Log")
        if len(st.session_state.prediction_history) == 0:
            st.warning("No dynamic evaluation entries compiled yet. Head to the 'Inference Console' and run predictions.")
        else:
            history_df = pd.DataFrame(st.session_state.prediction_history)
            st.dataframe(history_df, use_container_width=True)

            # Add Export Utility for research analysis reporting
            csv_data = history_df.to_csv(index=False).encode('utf-8')
            st.download_button(
                label="📥 Export Audit Run Data to CSV",
                data=csv_data,
                file_name=f"aqi_audit_run_{datetime.date.today()}.csv",
                mime="text/csv",
                use_container_width=True
            )

Overwriting app.py


In [ ]:
import joblib
from sklearn.preprocessing import MinMaxScaler
from xgboost import XGBRegressor
import numpy as np

print(" Generating and saving fallback ML assets...")
scaler_X = MinMaxScaler()
scaler_X.fit(np.random.rand(10, 7))
scaler_y = MinMaxScaler()
scaler_y.fit(np.random.rand(10, 1))
joblib.dump(scaler_X, 'scaler_X.pkl')
joblib.dump(scaler_y, 'scaler_y.pkl')
placeholder_model = XGBRegressor(n_estimators=100, learning_rate=0.5, random_state=42)
placeholder_model.fit(np.random.rand(10, 7), np.random.rand(10, 1))
joblib.dump(placeholder_model, 'xgboost_aqi_model.pkl')
print(" SUCCESS! The following files have been created in your directory:")

import os
print([f for f in os.listdir('.') if f.endswith('.pkl')])

 Generating and saving fallback ML assets...
 SUCCESS! The following files have been created in your directory:
['scaler_X.pkl', 'xgboost_aqi_model.pkl', 'scaler_y.pkl']


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
import numpy as np

dl_model = Sequential([
    Flatten(input_shape=(24, 7)),
    Dense(64, activation='relu'),
    Dense(1)
])

dl_model.compile(optimizer='adam', loss='mse')

dl_model.save('aqi_deep_learning_model.h5')


/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
